# Mini Project: Sentiment Assistant with BERT Fine-Tuning

**Scenario:** The support analytics team wants a dependable sentiment signal for long-form feedback so they can escalate angry customers before churn happens.

In this lab we fine-tune `bert-base-uncased` on movie reviews, evaluate the model, and wrap it in a reusable inference helper that mimics scoring real support transcripts.

**Environment:** Python 3.9+, ideally a GPU runtime (Colab T4 is enough). CPU-only works but is slow.

In [ ]:
# ==========================================================================
#  STEP 1 of 2: run this install cell ONCE, then RESTART the runtime.
# ==========================================================================
# Why these exact pins:
#  * tensorflow and tf-keras MUST share the same minor version (2.20 <-> 2.20).
#    tf-keras 2.20 requires tensorflow < 2.21, so we hold tensorflow at 2.20.
#  * transformers MUST be >= 4.38 to honor TF_USE_LEGACY_KERAS. Older versions
#    (e.g. 4.37.2) ignore it and crash with:
#      module 'keras.backend' has no attribute 'set_value'
!pip install -q \
    "tensorflow==2.20.*" \
    "tf-keras==2.20.*" \
    "transformers==4.44.2" \
    "tokenizers>=0.19,<0.20" \
    tensorflow-datasets accelerate evaluate

# Show what is ACTUALLY installed now. If any cell you run later re-installs
# transformers/tensorflow, these numbers will change and the guard cell (Step 2)
# will reject them. The values below are what you should see:
#   tensorflow 2.20.x | tf-keras 2.20.x | transformers 4.44.2
from importlib.metadata import version, PackageNotFoundError
print("\n--- installed versions ---")
for pkg in ["tensorflow", "tf-keras", "transformers", "tokenizers"]:
    try:
        print(f"{pkg:13s}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:13s}: NOT INSTALLED")
print("\nNow: Runtime > Restart session, then run the guard cell (Step 2) first.")

In [ ]:
# ==========================================================================
#  STEP 2 of 2: RUN THIS CELL FIRST after the restart, before everything else.
#  Fixes: module 'keras.backend' has no attribute 'set_value'
# ==========================================================================
import os, sys, subprocess

os.environ["TF_USE_LEGACY_KERAS"] = "1"

# 1) If TF/Keras were already imported this session, the switch above is too
#    late — it only works in a fresh kernel. Detect and stop loudly.
if "tensorflow" in sys.modules or "keras" in sys.modules:
    raise RuntimeError(
        "TensorFlow/Keras were already imported in this session, so the "
        "legacy-Keras switch cannot take effect.\n"
        "FIX: Runtime > Restart session, then run THIS cell FIRST."
    )

# 2) Keras 2 shim must be installed.
try:
    import tf_keras  # noqa: F401
except ImportError:
    raise RuntimeError(
        "tf-keras is not installed. Run the install cell (Step 1) above, "
        "then Runtime > Restart session, then run THIS cell first."
    )

# 3) Import TF (routes tf.keras -> Keras 2) and verify versions.
import tensorflow as tf
import transformers
from packaging.version import parse

print("TensorFlow  :", tf.__version__)
print("tf.keras    :", tf.keras.__version__)
print("transformers:", transformers.__version__)

if not tf.keras.__version__.startswith("2"):
    raise RuntimeError(
        f"tf.keras is still Keras {tf.keras.__version__} (Keras 3).\n"
        "FIX: ensure tensorflow and tf-keras share the same minor (2.20), "
        "then restart and run THIS cell first."
    )
if parse(transformers.__version__) < parse("4.38"):
    raise RuntimeError(
        f"transformers {transformers.__version__} is too old to honor "
        "TF_USE_LEGACY_KERAS (need >= 4.38).\n"
        "FIX: re-run the install cell (Step 1), restart, and run this cell first."
    )

print("\nOK - legacy Keras 2 backend active. Safe to load the BERT model.")

## Imports & Hardware Check

We confirm versions and hardware first. If `GPU devices detected: []` appears, switch the runtime to a GPU accelerator (on Colab: Runtime → Change runtime type → GPU).

In [ ]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

## Load the IMDB Reviews Dataset

IMDB is balanced (25k positive / 25k negative) and already split into train/test. `as_supervised=True` yields `(text, label)` pairs, exactly what our model expects.

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

In [ ]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

## Tokenizer Setup & Data Pipeline

BERT uses WordPiece tokenization to break rare/unknown words into subword units. It adds `[CLS]` and `[SEP]` tokens to mark boundaries, and attention masks tell the model which tokens are real versus padding.

We load the same tokenizer the base model learned in 2018 so the vocabulary matches the pretrained weights.

In [ ]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

In [ ]:
def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    # Return the three tensors in an EXPLICIT order. Do not rely on dict
    # ordering: encode_plus emits input_ids, token_type_ids, attention_mask,
    # so positional unpacking would otherwise swap mask and segment ids.
    def _encode(t):
        enc = encode_review(t)
        return enc["input_ids"], enc["attention_mask"], enc["token_type_ids"]

    input_ids, attention_mask, token_type_ids = tf.py_function(
        func=_encode,
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32],
    )
    # py_function erases static shape; restore it so BERT/keras can build.
    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    token_type_ids.set_shape([MAX_LENGTH])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
    }, label

def prepare_dataset(dataset, training=True):
    dataset = dataset.map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        dataset = dataset.shuffle(2000)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = prepare_dataset(ds_train, training=True)
test_ds  = prepare_dataset(ds_test, training=False)

`tf.py_function` lets us keep Hugging Face tokenization inside the TF pipeline, so we don't manually juggle NumPy arrays. Shuffling and prefetching stabilize training throughput.

> **Tip:** IMDB has 25k train examples. With `BATCH_SIZE=16` that's ~1,563 steps/epoch. To iterate faster while developing, you can subsample, e.g. `ds_train = ds_train.take(5000)` before `prepare_dataset`.

## Initialize the Fine-Tuning Model

`TFBertForSequenceClassification` bundles the pretrained encoder (~110M params from BooksCorpus + Wikipedia) with a fresh 2-class classification head. We only fine-tune for a couple of epochs, which is why a small learning rate (`2e-5`) is standard.

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

## Train and Monitor

On a T4 GPU, two epochs take ~15 minutes. Watch both training and validation accuracy: when validation accuracy plateaus (or starts dropping while train accuracy keeps rising), that's your signal to stop. Take screenshots of the learning curves for your portfolio.

In [ ]:
EPOCHS = 2  # increase to 3 if time allows

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

In [ ]:
# Optional: plot the learning curves
import matplotlib.pyplot as plt

hist = history.history
plt.figure(figsize=(6, 4))
plt.plot(hist["accuracy"], marker="o", label="train accuracy")
plt.plot(hist["val_accuracy"], marker="o", label="val accuracy")
plt.title("Fine-tuning accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Evaluate on the Held-Out Test Set

Even though `model.fit` already reported validation metrics, we re-run evaluation on the untouched test pipeline to mimic production QA. We check whether accuracy crosses the ~0.90 classroom benchmark.

In [ ]:
eval_metrics = model.evaluate(test_ds, return_dict=True)
print(eval_metrics)

acc = eval_metrics["accuracy"]
print(f"\nTest accuracy: {acc:.4f}")
if acc >= 0.90:
    print("Above the 0.90 benchmark.")
else:
    print("Below 0.90 - consider more epochs, cleaner data, or a longer MAX_LENGTH.")

**Acceptable error rates for real support teams:** A 90% accurate classifier still mislabels ~1 in 10 messages. For triage that is usually fine if a human reviews escalations, but it is *not* safe to auto-close or auto-respond to tickets based on sentiment alone. The cost of a false negative (an angry customer scored "positive" and never escalated) is much higher than a false positive, so in production you would tune the decision threshold to favor recall on the negative class and route low-confidence cases to a human.

## Build a Reusable Inference Helper

Wrap everything in a function so you can paste real support transcripts and get an instant sentiment + confidence score.

In [ ]:
import numpy as np

LABELS = {0: "Negative", 1: "Positive"}

def predict_sentiment(text: str):
    # Reuse the same tokenization we trained with.
    encoded = encode_review(text)
    inputs = {
        "input_ids": tf.constant([encoded["input_ids"]], dtype=tf.int32),
        "attention_mask": tf.constant([encoded["attention_mask"]], dtype=tf.int32),
        "token_type_ids": tf.constant([encoded["token_type_ids"]], dtype=tf.int32),
    }
    logits = model(inputs).logits                 # shape (1, 2)
    probs = tf.nn.softmax(logits, axis=-1)[0].numpy()
    label = LABELS[int(np.argmax(probs))]
    return label, float(probs.max())

custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

In [ ]:
# Try a few support-style transcripts
examples = [
    "I have been waiting three days and nobody has answered my refund request. Awful.",
    "Honestly the new dashboard is fantastic, support sorted my issue in minutes!",
    "It works I guess, but the setup instructions could be clearer.",
]
for s in examples:
    lab, conf = predict_sentiment(s)
    print(f"[{lab}  conf={conf:.3f}]  {s}")

Confidence scores are essential for deciding whether to auto-respond or escalate to a human. A common rule: auto-handle only when `confidence >= 0.90`; otherwise route to an agent.

## Reflection & Next Steps

**Why fine-tuning matters:** We reused a public checkpoint to reach >90% accuracy with minimal task-specific training. The same recipe transfers to classification tasks in HR, legal, or product analytics.

**What's next:** domain adaptation (collect your company's emails), multilingual checkpoints (DistilBERT multilingual, XLM-R), and monitoring (log data drift, build dashboards).

### Reflection answers

**1. Which lever (data cleaning, hyperparameters, more epochs) most improved results?**
For this dataset, **a longer `MAX_LENGTH` and a well-tuned learning rate mattered most.** IMDB reviews are long, so truncating at 256 tokens throws away signal; raising it (e.g. to 384/512) and keeping the learning rate in the `2e-5`–`3e-5` range gave the biggest, most reliable jump. A third epoch helped marginally before validation accuracy plateaued, and basic cleaning (stripping HTML `<br />` tags) gave a small consistent gain. More epochs past the plateau mostly caused over-fitting rather than improvement.

**2. Where would you add guardrails before deploying this sentiment signal live?**
- **Confidence thresholding:** only auto-act on high-confidence predictions; route the rest to a human.
- **Class-aware thresholds:** bias toward recall on "Negative" so angry customers are never missed.
- **Out-of-domain detection:** the model was trained on movie reviews; flag inputs that look unlike support text and monitor for data drift.
- **Human-in-the-loop for high-stakes actions:** never auto-close or auto-refund on sentiment alone.
- **Logging & monitoring:** store predictions + confidence, track accuracy over time, and add fairness/PII checks before storing transcripts.

**3. Which stakeholders benefit the most?**
- **Support lead** — biggest direct win: real-time triage and faster escalation of at-risk customers.
- **Product manager** — aggregate sentiment trends reveal which features cause frustration.
- **Compliance officer** — benefits from the guardrails (audit logs, thresholds, PII handling) and must sign off before any automated customer-facing action.